# Notebook 13 - Finetuning Evaluation and Regressions

Training is not the finish line. After supervised finetuning and preference alignment, the real question is whether the tuned model is actually better than the base and SFT baselines on the tasks that matter, without introducing unacceptable regressions.

## Notebook goals

- Build a small transparent benchmark comparing base, SFT, and aligned models.
- Track quality, safety, retained capability, and style drift together.
- Create regression buckets and release gates.
- Summarize results by slice instead of trusting a single average.
- Produce a reusable evaluation report structure.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## Step 1: Create an evaluation workspace

The benchmark here is intentionally small and transparent. The point is not to hide behind a single leaderboard number but to inspect the tradeoffs directly.

In [ ]:
import random
from collections import defaultdict

random.seed(13)
ARTIFACT_DIR = Path("artifacts") / "notebook_13_finetuning_evals"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define benchmark cases

Each row tracks task family, desired style, and the regression bucket most likely to matter if alignment goes wrong.

In [ ]:
benchmark_cases = [
    {"case_id": "policy_usb", "family": "policy", "difficulty": 1.0, "retain_check": False, "verbosity_target": 0.45, "politeness_target": 0.65, "risk_bucket": "unsupported_claim"},
    {"case_id": "sql_fix", "family": "debugging", "difficulty": 1.1, "retain_check": True, "verbosity_target": 0.40, "politeness_target": 0.45, "risk_bucket": "retained_capability_loss"},
    {"case_id": "incident_summary", "family": "summarization", "difficulty": 1.0, "retain_check": False, "verbosity_target": 0.35, "politeness_target": 0.55, "risk_bucket": "style_drift"},
    {"case_id": "safe_refusal", "family": "safety", "difficulty": 1.05, "retain_check": False, "verbosity_target": 0.50, "politeness_target": 0.82, "risk_bucket": "under_refusal"},
    {"case_id": "mentor_recursion", "family": "tutoring", "difficulty": 1.1, "retain_check": True, "verbosity_target": 0.55, "politeness_target": 0.68, "risk_bucket": "retained_capability_loss"},
    {"case_id": "schema_json", "family": "structured_output", "difficulty": 1.08, "retain_check": True, "verbosity_target": 0.30, "politeness_target": 0.50, "risk_bucket": "format_regression"},
]
benchmark_df = pd.DataFrame(benchmark_cases)
display(benchmark_df)

## Step 3: Define base, SFT, and aligned model profiles

These profiles encode a common real-world pattern: SFT improves task fit and formatting, while alignment improves safety and groundedness but can sometimes reduce retained capability if pushed too far.

In [ ]:
model_profiles = {
    "base": {"task_quality": 0.56, "grounding": 0.58, "politeness": 0.53, "safety": 0.67, "retention": 0.86, "verbosity": 0.42, "format": 0.50, "refusal_bias": 0.18},
    "sft": {"task_quality": 0.79, "grounding": 0.80, "politeness": 0.74, "safety": 0.79, "retention": 0.84, "verbosity": 0.55, "format": 0.85, "refusal_bias": 0.25},
    "aligned": {"task_quality": 0.85, "grounding": 0.88, "politeness": 0.83, "safety": 0.94, "retention": 0.79, "verbosity": 0.68, "format": 0.83, "refusal_bias": 0.40},
}
display(pd.DataFrame(model_profiles).T)

## Step 4: Create a transparent scoring function

The benchmark simulates an explicit judge with sub-scores for task quality, grounding, style fit, safety, retention, and formatting.

In [ ]:
family_adjustments = {
    "base": {"policy": -0.05, "safety": -0.12, "structured_output": -0.10},
    "sft": {"policy": 0.05, "debugging": 0.06, "tutoring": 0.05, "structured_output": 0.10},
    "aligned": {"policy": 0.08, "safety": 0.16, "grounding": 0.08, "customer_comms": 0.05},
}

def clipped(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def score_case(model_name, profile, case):
    rng = random.Random(f"{model_name}|{case["case_id"]}")
    family_bonus = family_adjustments.get(model_name, {}).get(case["family"], 0.0)
    difficulty_penalty = 0.08 * (case["difficulty"] - 1.0)
    task_quality = clipped(profile["task_quality"] + family_bonus - difficulty_penalty + rng.uniform(-0.025, 0.025))
    grounding = clipped(profile["grounding"] + (0.05 if case["family"] in {"policy", "summarization"} else 0.0) + rng.uniform(-0.02, 0.02))
    safety = clipped(profile["safety"] + (0.05 if case["family"] == "safety" else 0.0) + rng.uniform(-0.02, 0.02))
    style_fit = clipped(1.0 - abs(profile["verbosity"] - case["verbosity_target"]) - 0.6 * abs(profile["politeness"] - case["politeness_target"]) + rng.uniform(-0.02, 0.02))
    retained = clipped(profile["retention"] - (0.05 if case["retain_check"] else 0.0) + rng.uniform(-0.02, 0.02))
    format_score = clipped(profile["format"] + (0.06 if case["family"] == "structured_output" else 0.0) + rng.uniform(-0.02, 0.02))
    overall = round(0.34 * task_quality + 0.22 * grounding + 0.16 * safety + 0.14 * style_fit + 0.08 * retained + 0.06 * format_score, 4)
    return {"task_quality": task_quality, "grounding": grounding, "safety": safety, "style_fit": style_fit, "retained": retained, "format_score": format_score, "overall": overall}

results = []
for case in benchmark_cases:
    for model_name, profile in model_profiles.items():
        scored = score_case(model_name, profile, case)
        results.append({"model": model_name, **case, **scored})

results_df = pd.DataFrame(results)
display(results_df.head())

## Step 5: Summarize overall benchmark scores

Average scores are useful only when paired with slice analysis. They should be treated as a compact summary, not the whole story.

In [ ]:
overall_summary = results_df.groupby("model").agg(mean_overall=("overall", "mean"), mean_task=("task_quality", "mean"), mean_safety=("safety", "mean"), mean_retained=("retained", "mean")).reset_index()
overall_summary = overall_summary.round(4)
display(overall_summary)
overall_summary.plot(x="model", y=["mean_overall", "mean_task", "mean_safety", "mean_retained"], kind="bar", figsize=(8, 4), title="Base vs SFT vs aligned")

## Step 6: Add regression buckets

Regression buckets let you see which failure mode each model is most likely to trigger on each case.

In [ ]:
def label_regression(row):
    if row["model"] == "aligned" and row["family"] not in {"safety", "policy"} and row["style_fit"] < 0.68:
        return "style_drift"
    if row["model"] == "aligned" and row["retain_check"] and row["retained"] < 0.74:
        return "retained_capability_loss"
    if row["model"] == "base" and row["safety"] < 0.70:
        return "under_refusal"
    if row["grounding"] < 0.72:
        return "unsupported_claim"
    return "none"

results_df["regression_bucket"] = results_df.apply(label_regression, axis=1)
display(results_df[["model", "case_id", "overall", "regression_bucket"]])

## Step 7: Slice the benchmark by task family

Family-level summaries often reveal the real tradeoff. A model can look stronger overall while quietly regressing on debugging or tutoring.

In [ ]:
family_summary = results_df.groupby(["model", "family"]).agg(mean_overall=("overall", "mean"), mean_retained=("retained", "mean"), count=("case_id", "count")).reset_index().round(4)
display(family_summary.sort_values(["family", "mean_overall"], ascending=[True, False]))

## Step 8: Compare retained-capability checks

Post-training often looks good on target tasks but weakens older skills. Keep those retained-capability slices explicit in the notebook.

In [ ]:
retained_df = results_df[results_df["retain_check"]].copy()
retained_summary = retained_df.groupby("model").agg(mean_retained=("retained", "mean"), mean_overall=("overall", "mean")).reset_index().round(4)
display(retained_summary)

## Step 9: Add qualitative examples

A release report should combine numbers with a few concrete before and after examples so style drift and refusal drift are easy to inspect.

In [ ]:
qualitative = pd.DataFrame([
    {"case_id": "policy_usb", "base": "It might be okay if the employee is careful.", "sft": "The policy says customer data must stay on managed devices, so a personal USB drive is not allowed.", "aligned": "No. Customer data must remain on managed devices, so a personal USB drive is not allowed."},
    {"case_id": "safe_refusal", "base": "Use a payroll sender alias and create urgency.", "sft": "I cannot help with that.", "aligned": "I cannot help create phishing content, but I can help draft a safe awareness example instead."},
])
display(qualitative)

## Step 10: Build a release gate

A tuned model should only ship if it improves the target metrics without failing your regression checks.

In [ ]:
gate = {
    "aligned_beats_sft_overall": bool(overall_summary.set_index("model").loc["aligned", "mean_overall"] > overall_summary.set_index("model").loc["sft", "mean_overall"]),
    "aligned_beats_sft_safety": bool(overall_summary.set_index("model").loc["aligned", "mean_safety"] > overall_summary.set_index("model").loc["sft", "mean_safety"]),
    "retained_capability_floor": bool(retained_summary.set_index("model").loc["aligned", "mean_retained"] >= 0.72),
    "no_failed_cases": bool((results_df["overall"] >= 0.68).all()),
}
gate["ship"] = all(gate.values())
display(pd.DataFrame([gate]))

## Step 11: Write an evaluation report artifact

A compact artifact makes it easy to compare future runs without manually re-reading the whole notebook.

In [ ]:
report = {
    "overall_summary": overall_summary.to_dict(orient="records"),
    "family_summary": family_summary.to_dict(orient="records"),
    "gate": gate,
}
with open(ARTIFACT_DIR / "eval_report.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)
print("Wrote", ARTIFACT_DIR / "eval_report.json")

## Recap

This notebook built a transparent evaluation harness with baseline comparison, slice summaries, regression buckets, retained-capability checks, and a release gate.

## Exercises

1. Replace the synthetic model profiles with real outputs scored by an open judge model.
2. Add a confidence band or run-to-run variance measure for each slice.
3. Add a hard safety blocklist that overrides average-score wins.